La libreria **sklearn.pipeline** offre strumenti per concatenare più passaggi di elaborazione (trasformatori) e un modello finale (estimatore) in un unico oggetto. Questo approccio, noto come "pipeline", semplifica il flusso di lavoro, previene errori comuni e rende il codice più pulito e riproducibile.


Una pipeline in Scikit-learn è un oggetto che incapsula una sequenza di trasformazioni dei dati e un estimatore finale. Quando si chiama il metodo `.fit()` sulla pipeline, i dati vengono passati attraverso ogni passaggio in sequenza: ogni trasformatore viene addestrato (`fit_transform`) sui dati di input e l'output viene passato al passaggio successivo, fino a raggiungere l'estimatore finale, che viene addestrato sui dati trasformati.

I principali vantaggi dell'utilizzo di una pipeline sono:
-   **Convenienza**: Raggruppa più passaggi in un unico oggetto, semplificando l'addestramento e la valutazione.
-   **Prevenzione del Data Leakage**: Garantisce che i passaggi di pre-processing (come la scalatura) vengano addestrati solo sui dati di training, evitando che informazioni del set di test "trapelino" nel processo di addestramento.
-   **Ottimizzazione degli Iperparametri**: Facilita la ricerca a griglia (Grid Search) simultanea degli iperparametri di tutti i passaggi della pipeline.

In questo esempio, creeremo una pipeline che esegue due passaggi:
1.  **Standardizzazione dei dati**: Utilizza `StandardScaler` per scalare le feature.
2.  **Classificazione**: Utilizza un classificatore `SVC` (Support Vector Classifier) per effettuare le predizioni.

In [1]:
# Esempio: Creazione di una Pipeline di Classificazione
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Genera dati di esempio
X, y = make_classification(random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# 2. Definisci i passaggi della pipeline
# Ogni passaggio è una tupla ('nome_passaggio', oggetto_stimatore)
steps = [
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear', random_state=42))
]

# 3. Crea la pipeline
pipeline = Pipeline(steps)

# 4. Addestra la pipeline
# StandardScaler viene addestrato e trasforma i dati, poi SVC viene addestrato sui dati trasformati
pipeline.fit(X_train, y_train)

# 5. Esegui le predizioni
y_pred = pipeline.predict(X_test)

# 6. Valuta il modello
accuracy = pipeline.score(X_test, y_test)

print(f"Accuracy della pipeline: {accuracy:.2f}")

Accuracy della pipeline: 1.00


Uno dei maggiori vantaggi delle pipeline è la loro integrazione con `GridSearchCV` per l'ottimizzazione degli iperparametri. È possibile definire una griglia di parametri per i diversi passaggi della pipeline, specificando il nome del passaggio seguito da `__` e il nome del parametro.

In [2]:
from sklearn.model_selection import GridSearchCV

# Definisci la griglia di parametri per la pipeline
# 'svc__C' si riferisce al parametro C del passaggio 'svc'
param_grid = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf']
}

# Crea e addestra GridSearchCV con la pipeline
search = GridSearchCV(pipeline, param_grid, cv=5)
search.fit(X_train, y_train)

print("Migliori parametri trovati:", search.best_params_)
print(f"Migliore accuracy con Grid Search: {search.best_score_:.2f}")

Migliori parametri trovati: {'svc__C': 0.1, 'svc__kernel': 'linear'}
Migliore accuracy con Grid Search: 0.96
